In [ ]:
!pip install evaluate
!pip install huggingface_hub
!pip install chromadb
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Foun

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
header1 = "Question: "
header2 = "Passage: "
header3 = "Answer: "

column_names1 = "question"
column_names2 = "passage"
column_names3 = "answer"

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
    global column_names1, column_names2, column_names3, header1, header2, header3
    column_names1 = col1
    column_names2 = col2
    column_names3 = col3
    header1 = header_val1
    header2 = header_val2
    header3 = header_val3

In [ ]:
# load dataset — same as baseline
raw_datasets = load_dataset("flowaicom/HaluEval")

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets = raw_datasets.map(label_map)
print(f"Dataset loaded: {len(raw_datasets['test'])} rows")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset loaded: 10000 rows


In [ ]:
set_master_columns("question", "passage", "answer", "Question: ", "Passage: ", "Answer: ")

split_datasets = raw_datasets["test"].train_test_split(test_size=0.2, seed=42)

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset  = split_datasets["test"].shuffle(seed=42).select(range(100))

print(f"Train size: {len(small_train_dataset)}")
print(f"Eval size:  {len(small_eval_dataset)}")

Train size: 400
Eval size:  100


In [ ]:
# mount Google Drive so ChromaDB


from google.colab import drive
drive.mount('/content/drive')

CHROMA_PATH    = "/content/drive/MyDrive/chroma_db_rag"
RETRIEVE_TOP_K = 1

print("Loading embedding model...")
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection("halueval_passages")

Mounted at /content/drive
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ----------------------------------------
#build passage store from ALL
# 10,000 rows for realistic retrieval.
# ----------------------------------------

# use all 10,000 passages as the retrieval pool
all_passages = list(raw_datasets["test"]["passage"])
all_ids = [f"passage_{i}" for i in range(len(all_passages))]

def build_passage_store():
    existing = collection.get()
    already_stored = len(existing["ids"])

    if already_stored >= len(all_passages):
        print(f"Passage store complete ({already_stored} passages), skipping...")
        return

    # resume from where we left off
    start_index = already_stored
    remaining_passages = all_passages[start_index:]
    remaining_ids = all_ids[start_index:]

    print(f"Building passage store from passage {start_index} / {len(all_passages)}...")

    batch_size = 64
    for i in range(0, len(remaining_passages), batch_size):
        batch = remaining_passages[i:i+batch_size]
        batch_ids = remaining_ids[i:i+batch_size]
        embeddings = embedder.encode(batch, normalize_embeddings=True).tolist()
        collection.add(
            documents=batch,
            embeddings=embeddings,
            ids=batch_ids
        )
        print(f"Stored {start_index + min(i+batch_size, len(remaining_passages))}/{len(all_passages)} passages")

    print("Passage store ready!")

build_passage_store()

Building passage store from passage 0 / 10000...
  Stored 64/10000 passages
  Stored 128/10000 passages
  Stored 192/10000 passages
  Stored 256/10000 passages
  Stored 320/10000 passages
  Stored 384/10000 passages
  Stored 448/10000 passages
  Stored 512/10000 passages
  Stored 576/10000 passages
  Stored 640/10000 passages
  Stored 704/10000 passages
  Stored 768/10000 passages
  Stored 832/10000 passages
  Stored 896/10000 passages
  Stored 960/10000 passages
  Stored 1024/10000 passages
  Stored 1088/10000 passages
  Stored 1152/10000 passages
  Stored 1216/10000 passages
  Stored 1280/10000 passages
  Stored 1344/10000 passages
  Stored 1408/10000 passages
  Stored 1472/10000 passages
  Stored 1536/10000 passages
  Stored 1600/10000 passages
  Stored 1664/10000 passages
  Stored 1728/10000 passages
  Stored 1792/10000 passages
  Stored 1856/10000 passages
  Stored 1920/10000 passages
  Stored 1984/10000 passages
  Stored 2048/10000 passages
  Stored 2112/10000 passages
  Stored 2

In [ ]:
def retrieve_passage(question):
    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=RETRIEVE_TOP_K
    )
    return results["documents"][0][0]  # top-1 passage

In [ ]:

# replace true passages with retrieved passages in the subset


def replace_passage(example):
    example["passage"] = retrieve_passage(example["question"])
    return example

print("Replacing passages with retrieved passages...")
rag_train_dataset = small_train_dataset.map(replace_passage)
rag_eval_dataset  = small_eval_dataset.map(replace_passage)
print("Done!")

Parameter 'function'=<function replace_passage at 0x7acbd00c6ac0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Replacing passages with retrieved passages...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Done!


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
MAX_LENGTH = 256

def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (header1 + q, header2 + p + header3 + a)
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]
    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

rag_train_tokenized = rag_train_dataset.map(tokenize_function, batched=True)
rag_eval_tokenized  = rag_eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# model and metrics
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall  = evaluate.load("recall")
f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

# same training args as baseline
training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs=8, learning_rate=2e-5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# train and evaluate
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=rag_train_tokenized,
    eval_dataset=rag_eval_tokenized,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.667833,0.670000,0.833333,0.408163,0.547945
2,No log,0.321186,0.940000,0.890909,1.000000,0.942308
3,No log,0.204935,0.960000,0.924528,1.000000,0.960784
4,No log,0.215613,0.960000,0.924528,1.000000,0.960784
5,No log,0.361170,0.940000,0.890909,1.000000,0.942308
6,No log,0.287983,0.950000,0.907407,1.000000,0.951456
7,No log,0.264138,0.960000,0.924528,1.000000,0.960784
8,No log,0.261293,0.960000,0.924528,1.000000,0.960784


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.2612934410572052,
 'eval_accuracy': 0.96,
 'eval_precision': 0.9245283018867925,
 'eval_recall': 1.0,
 'eval_f1': 0.9607843137254902,
 'eval_runtime': 1.6048,
 'eval_samples_per_second': 62.313,
 'eval_steps_per_second': 8.101,
 'epoch': 8.0}

In [ ]:

# Optional: swap Question and Passage order


set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")

rag_train_tokenized = rag_train_dataset.map(tokenize_function, batched=True)
rag_eval_tokenized  = rag_eval_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs=8, learning_rate=2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=rag_train_tokenized,
    eval_dataset=rag_eval_tokenized,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()